# Normalization and Spectral Whitening

## Mohit Bokariya
## mohit.seismology@gmail.com

In [83]:
import numpy as np
from obspy import read
from obspy.signal.util import next_pow_2
from natsort import natsorted
import os


### Functions

In [84]:
def one_bit_normalize(trace):
    """Apply one-bit normalization to a trace."""
    return np.sign(trace)

def spectral_whitening(trace, dt):
    """Apply spectral whitening in the frequency domain."""
    n = next_pow_2(len(trace))
    spec = np.fft.rfft(trace, n=n)
    amp = np.abs(spec)
    amp[amp == 0] = 1  # prevent division by zero
    white_spec = spec / amp
    white_trace = np.fft.irfft(white_spec, n=n)
    return white_trace[:len(trace)]

def process_sac_file(file_path):
    try:
        st = read(file_path)
        tr = st[0]
        tr.data = one_bit_normalize(tr.data)
        tr.data = spectral_whitening(tr.data, tr.stats.delta)
        tr.write(file_path, format='SAC')  # overwrite file
        print(f"Processed: {file_path}")
    except Exception as e:
        print(f"Failed to process {file_path}: {e}")

In [85]:
#Directory of Network which have folders of Stations (Input) 
Main_Path="/home/sig/Mohit/Data/RASPRAW/ZTEST/M50_SAC/"

In [88]:
# If we have a lot of stations use this
#Station_list = natsorted(os.listdir(Main_Path)) 

                #OR

#For specific Stations use this
Station_list=["CFTR","CYBR", "GAT2", "GRVT", "GUST", "KIDZ", "LBRY", "STOR", "SUDE", "TSNM"]


print(" Staion names :",Station_list, "\n","Total no. of Station :", len(Station_list))


 Staion names : ['CFTR', 'CYBR', 'GAT2', 'GRVT', 'GUST', 'KIDZ', 'LBRY', 'STOR', 'SUDE', 'TSNM'] 
 Total no. of Station : 10


In [ ]:
for i in range(len(Station_list)):
    STN=Station_list[i]    
    STN_Path=os.path.join(Main_Path, STN)
    contents = os.listdir(STN_Path)
    FILES=natsorted(contents)
    
    #print("STATION :" ,STN)
    
    for file in FILES:
        File_Path=os.path.join(STN_Path,file)
        #print(File_Path)
        process_sac_file(File_Path)

# Shorter Version

In [ ]:
import os
import numpy as np
from obspy import read
from obspy.signal.util import next_pow_2

def one_bit_normalize(trace):
    """Apply one-bit normalization to a trace."""
    return np.sign(trace)

def spectral_whitening(trace, dt):
    """Apply spectral whitening in the frequency domain."""
    n = next_pow_2(len(trace))
    spec = np.fft.rfft(trace, n=n)
    amp = np.abs(spec)
    amp[amp == 0] = 1  # prevent division by zero
    white_spec = spec / amp
    white_trace = np.fft.irfft(white_spec, n=n)
    return white_trace[:len(trace)]

def process_sac_file(file_path):
    try:
        st = read(file_path)
        tr = st[0]
        tr.data = one_bit_normalize(tr.data)
        tr.data = spectral_whitening(tr.data, tr.stats.delta)
        tr.write(file_path, format='SAC')  # overwrite file
        print(f"Processed: {file_path}")
    except Exception as e:
        print(f"Failed to process {file_path}: {e}")

def process_all_sac_files_alphabetical(root_dir):
    # Sort folders alphabetically
    for root, dirs, files in sorted(os.walk(root_dir)):
        dirs.sort()  # Sort subdirectories in-place
        # Sort and process SAC files alphabetically
        for file in sorted(files):
            if file.lower().endswith(".sac"):
                full_path = os.path.join(root, file)
                process_sac_file(full_path)

# === Run Processing ===
if __name__ == "__main__":
    root_directory = "/home/sig/Mohit/Data/NewRASP/ZTEST/Z50_SAC/"  # Replace this your path
    process_all_sac_files_alphabetical(root_directory)
